# IEEE-CIS Fraud Detection — Model Training

Pipeline:
1. Load & clean data
2. Feature engineering (20+ features)
3. SMOTE balancing
4. Train XGBoost + Isolation Forest ensemble
5. Evaluate (AUC-ROC, Precision, Recall, F1)
6. SHAP explainability
7. Save artefacts → `model/fraud_model.pkl`, `model/scaler.pkl`

> **Write down your AUC-ROC after Step 5 — it goes on your resume.**

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                              f1_score, classification_report, RocCurveDisplay)
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

print('Libraries loaded.')

## 1. Load Data

In [ ]:
DATA_PATH = Path('../data/raw/train_transaction.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Rows: {len(df):,}  |  Fraud rate: {df["isFraud"].mean()*100:.2f}%')

## 2. Feature Engineering (mirrors api/preprocessing.py)

In [ ]:
HIGH_RISK_DOMAINS = {
    'anonymous.com', 'mail.com', 'protonmail.com',
    'guerrillamail.com', 'throwam.com', 'yopmail.com'
}
PRODUCT_CODES = {'W': 0, 'H': 1, 'C': 2, 'S': 3, 'R': 4}

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    d = pd.DataFrame()

    d['log_transaction_amt']    = np.log1p(df['TransactionAmt'])
    amt_mean = df['TransactionAmt'].mean()
    amt_std  = df['TransactionAmt'].std()
    print(f'  TRAIN_AMT_MEAN = {amt_mean:.2f}  TRAIN_AMT_STD = {amt_std:.2f}')
    print('  --> Update these in api/preprocessing.py!')
    d['amt_zscore']             = (df['TransactionAmt'] - amt_mean) / (amt_std + 1e-9)

    hour                        = (df['TransactionDT'] % 86400) / 3600
    d['time_of_day_bucket']     = (hour / 4).astype(int)
    d['is_night_txn']           = ((hour >= 22) | (hour < 6)).astype(int)
    days                        = df['TransactionDT'] / 86400
    d['is_weekend']             = (days.astype(int) % 7 >= 5).astype(int)
    d['days_since_ref']         = days
    d['velocity_1h_flag']       = 0   # placeholder for real-time feature

    email                       = df['P_emaildomain'].fillna('').str.lower()
    d['high_risk_email_domain'] = email.isin(HIGH_RISK_DOMAINS).astype(int)

    card_type                   = df.get('card_type', pd.Series([''] * len(df))).fillna('')
    d['is_credit']              = (card_type.str.lower() == 'credit').astype(int)
    d['product_cd_encoded']     = df['ProductCD'].map(PRODUCT_CODES).fillna(0)

    d['addr1_missing']          = df['addr1'].isnull().astype(int)
    d['dist1_log']              = np.log1p(df['dist1'].fillna(0))
    d['dist1_missing']          = df['dist1'].isnull().astype(int)

    c1                          = df['C1'].fillna(1)
    c2                          = df['C2'].fillna(1)
    d['c_ratio']                = c1 / (c2 + 1)

    d['d1_missing']             = df['D1'].isnull().astype(int)
    d['d15_missing']            = df['D15'].isnull().astype(int)

    d['card1_norm']             = df['card1'].fillna(9500) / 18396.0
    d['card2_norm']             = df['card2'].fillna(111) / 600.0

    d['v258_flag']              = (df['V258'].fillna(0) > 0).astype(int)
    d['v308_flag']              = (df['V308'].fillna(0) > 0).astype(int)

    d['amt_x_c1']               = df['TransactionAmt'] * c1
    d['amt_bin']                = (df['TransactionAmt'] / 50).clip(upper=4).astype(int)

    return d

X = engineer_features(df)
y = df['isFraud'].values
print(f'\nFeature matrix shape: {X.shape}')
X.head(3)

## 3. Train/Test Split + Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 4. SMOTE Balancing (training set only)

In [ ]:
print(f'Before SMOTE: {np.bincount(y_train)}')
sm = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
print(f'After  SMOTE: {np.bincount(y_res)}')

## 5. Train XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1,
    eval_metric='auc',
    early_stopping_rounds=30,
    use_label_encoder=False,
    random_state=42,
    tree_method='hist',   # fast for large datasets
    device='cpu',
)

xgb_model.fit(
    X_res, y_res,
    eval_set=[(X_test_sc, y_test)],
    verbose=50,
)
print('XGBoost training complete.')

## 6. Train Isolation Forest (anomaly detector)

In [ ]:
# Fit on legit transactions only — learns the "normal" boundary
legit_mask = y_train == 0
iso = IsolationForest(
    n_estimators=200,
    contamination=0.035,   # ~3.5% fraud rate
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_train_sc[legit_mask])
print('Isolation Forest training complete.')

## 7. Evaluate Ensemble

In [ ]:
# XGBoost probabilities
xgb_proba = xgb_model.predict_proba(X_test_sc)[:, 1]

# Isolation Forest: sigmoid-normalise anomaly scores
iso_scores = iso.decision_function(X_test_sc)
iso_proba  = 1 / (1 + np.exp(iso_scores))

# Ensemble: weighted average
ensemble_proba = 0.75 * xgb_proba + 0.25 * iso_proba
ensemble_pred  = (ensemble_proba >= 0.5).astype(int)

auc      = roc_auc_score(y_test, ensemble_proba)
prec     = precision_score(y_test, ensemble_pred)
rec      = recall_score(y_test, ensemble_pred)
f1       = f1_score(y_test, ensemble_pred)

print('=' * 45)
print('  ENSEMBLE METRICS (XGBoost 75% + IF 25%)')
print('=' * 45)
print(f'  AUC-ROC   : {auc:.4f}   <-- PUT THIS ON YOUR RESUME')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1        : {f1:.4f}')
print('=' * 45)
print()
print(classification_report(y_test, ensemble_pred, target_names=['Legit', 'Fraud']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, ensemble_proba, ax=ax, name='Ensemble')
RocCurveDisplay.from_predictions(y_test, xgb_proba,       ax=ax, name='XGBoost only')
ax.set_title('ROC Curve — Fraud Detection Ensemble')
plt.tight_layout()
plt.savefig('../data/raw/roc_curve.png')
plt.show()

## 8. SHAP Feature Importance

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
# Use a sample for speed
sample_idx  = np.random.choice(len(X_test_sc), size=2000, replace=False)
shap_values = explainer.shap_values(X_test_sc[sample_idx])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[sample_idx], feature_names=X.columns.tolist(), show=False)
plt.tight_layout()
plt.savefig('../data/raw/shap_summary.png')
plt.show()

## 9. Save Artefacts

In [ ]:
MODEL_DIR = Path('../model')
MODEL_DIR.mkdir(exist_ok=True)

artefacts = {'xgb': xgb_model, 'iso': iso}
joblib.dump(artefacts, MODEL_DIR / 'fraud_model.pkl', compress=3)
joblib.dump(scaler,    MODEL_DIR / 'scaler.pkl',      compress=3)

print('Saved:')
print(f'  {MODEL_DIR}/fraud_model.pkl')
print(f'  {MODEL_DIR}/scaler.pkl')
print()
print(f'  Final AUC-ROC = {auc:.4f}  --> update api/config.py MODEL_VERSION if retraining')